# DINOv3 deep fine-tuning: selectable backbone and crop dataset

One notebook trains either **DINOv3 ViT-B/16** or **DINOv3 ConvNeXt-B** on either **wine-label crops** or **whole-bottle crops**. Change only `BACKBONE` and `DATASET_KIND` in the first code cell. Both models load strictly from attached local safetensors; no model hub is used.

Schedule: heads (5 epochs) → last 4 backbone blocks (5 epochs) → fully unfrozen backbone (up to 100 epochs). Checkpoint selection and stage-3 early stopping use **val_unseen Recall@1**.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

# ========================= USER PARAMETERS =========================
BACKBONE = 'vitb16'      # 'vitb16' or 'convnextb'
DATASET_KIND = 'labels'  # 'labels' or 'bottles'
RESUME_CHECKPOINT = None # None = fresh start from the selected safetensors
RUN_SMOKE_TEST = True
KAGGLE_USERNAME = 'konstantinmelnikof'
# ==================================================================

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr')
BACKBONE_SPECS = {
    'vitb16': {
        'display_name': 'DINOv3 ViT-B/16',
        'weights_slug': 'viscaner-dinov3-vitb16-weights',
        'weights_filename': 'model.safetensors',
    },
    'convnextb': {
        'display_name': 'DINOv3 ConvNeXt-B',
        'weights_slug': 'viscaner-dinov3-vitb16-weights',
        'weights_filename': 'dinov3-convnext-b.safetensors',
    },
}
DATASET_SPECS = {
    'labels': {
        'display_name': 'wine-label crops',
        'slug': 'viscaner-dinov3-data',
        'metadata': 'crops_metadata.csv',
    },
    'bottles': {
        'display_name': 'whole-bottle crops',
        'slug': 'viscaner-bottle-classifier-data',
        'metadata': 'crops_metadata.csv',
    },
}
if BACKBONE not in BACKBONE_SPECS:
    raise ValueError(f'BACKBONE must be one of {tuple(BACKBONE_SPECS)}')
if DATASET_KIND not in DATASET_SPECS:
    raise ValueError(f'DATASET_KIND must be one of {tuple(DATASET_SPECS)}')
model_spec = BACKBONE_SPECS[BACKBONE]
data_spec = DATASET_SPECS[DATASET_KIND]
RUN_NAME = f'dinov3_deeptune_{BACKBONE}_{DATASET_KIND}_unseen_r1'

print(
    f'SETUP | backbone={BACKBONE} ({model_spec["display_name"]}) | '
    f'dataset={DATASET_KIND} ({data_spec["display_name"]})',
    flush=True,
)

def dataset_root(slug: str) -> Path:
    candidates = (
        INPUT / slug,
        INPUT / 'datasets' / KAGGLE_USERNAME / slug,
    )
    matches = [path for path in candidates if path.is_dir()]
    if len(matches) != 1:
        raise RuntimeError(
            f'Expected one attached Kaggle dataset {slug}. Checked: '
            + ', '.join(str(path) for path in candidates)
        )
    return matches[0]

def require_file(root: Path, relative: str) -> Path:
    path = root / relative
    if not path.is_file():
        raise FileNotFoundError(f'Missing required file: {path}')
    return path

def run_streaming(command: list[str], log_path: Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            command, cwd=PROJECT, env=environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        exit_code = process.wait()
    if exit_code != 0:
        raise RuntimeError(f'Process failed with exit code {exit_code}. See {log_path}')

print('SETUP | resolving attached datasets...', flush=True)
code_root = dataset_root('viscaner-dinov3-code')
data_root = dataset_root(data_spec['slug'])
weights_root = dataset_root(model_spec['weights_slug'])
metadata = require_file(data_root, data_spec['metadata'])
manifest = require_file(data_root, 'bottle_images_manifest.csv')
weights = require_file(weights_root, model_spec['weights_filename'])
if not (data_root / 'crops').is_dir() or not (data_root / 'refs').is_dir():
    raise FileNotFoundError(f'{data_root} must contain crops/ and refs/')

print('SETUP | copying the versioned code dataset...', flush=True)
shutil.copytree(
    code_root, PROJECT, dirs_exist_ok=True,
    ignore=shutil.ignore_patterns('dataset-metadata.json'),
)
checksum_path = PROJECT / 'code_sha256.json'
if not checksum_path.is_file():
    raise FileNotFoundError('Update viscaner-dinov3-code: code_sha256.json is missing')
checksums = json.loads(checksum_path.read_text(encoding='utf-8'))
for filename, expected in checksums.items():
    candidate = PROJECT / filename
    actual = hashlib.sha256(candidate.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code integrity check failed: {filename}')
print(f'SETUP | verified {len(checksums)} code files', flush=True)

required_code = (
    PROJECT / 'dinov3_retrieval.py',
    PROJECT / 'full_finetune.py',
    PROJECT / 'deeptune_backbones.py',
    PROJECT / 'train_dinov3_deeptune.py',
)
for path in required_code:
    if not path.is_file():
        raise FileNotFoundError(f'Attached code dataset is outdated: {path.name} is missing')

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print('SETUP | installing Python dependencies...', flush=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=5.6,<6', 'safetensors>=0.5,<1',
    'scikit-learn>=1.5,<2', 'PyYAML>=6,<7',
], check=True)
print('SETUP | dependencies installed', flush=True)

import pandas as pd
import torch
import transformers
import yaml
from deeptune_backbones import MODEL_VARIANTS, validate_weights

if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before running this notebook')
if 'wine_slug' not in pd.read_csv(metadata, nrows=1).columns:
    raise ValueError(f'{metadata} has no wine_slug column')
validate_weights(weights, BACKBONE)
print(f'SETUP | weights SHA256 verified: {weights.name}', flush=True)
print('CUDA:', torch.cuda.get_device_name(0), 'GPU count:', torch.cuda.device_count(), flush=True)
print('torch:', torch.__version__, 'transformers:', transformers.__version__, flush=True)

config_path = PROJECT / 'configs/dinov3_full_finetune.yaml'
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
work_data = PROJECT / 'datasets' / f'dinov3_{BACKBONE}_{DATASET_KIND}'
cfg.update(
    project_root=str(PROJECT),
    weights_path=str(weights),
    crops_metadata_path=str(metadata),
    bottle_manifest_path=str(manifest),
    crops_root=str(data_root / 'crops'),
    refs_root=str(data_root / 'refs'),
    index_path=str(work_data / 'index.csv'),
    split_summary_path=str(work_data / 'split_summary.json'),
    models_dir=str(PROJECT / 'models' / RUN_NAME),
    runs_dir=str(PROJECT / 'runs/dinov3_retrieval'),
    run_name=RUN_NAME,
    device='cuda',
    amp=True,
)
assert cfg['stage1_epochs'] == 5 and cfg['stage2_epochs'] == 5
assert cfg['stage3_epochs'] == 100 and cfg['unfreeze_last_blocks'] == 4
assert cfg['early_stopping_split'] == 'val_unseen'
assert cfg['early_stopping_metric'] == 'recall_at_1'
if RESUME_CHECKPOINT is None and any(Path(cfg['models_dir']).glob('*.pt')):
    raise RuntimeError('Fresh run found checkpoints. Use a clean session or change RUN_NAME.')
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('SETUP COMPLETE | configuration:', flush=True)
print(json.dumps(cfg, indent=2), flush=True)


## CUDA smoke test

A disposable model checks strict checkpoint loading, forward/backward, and all three trainability phases before the long run.


In [ ]:
if RUN_SMOKE_TEST:
    smoke_cfg = dict(cfg)
    smoke_cfg.update(
        models_dir=str(PROJECT / 'smoke/models'),
        runs_dir=str(PROJECT / 'smoke/runs'),
        run_name=f'deeptune_smoke_{BACKBONE}_{DATASET_KIND}',
    )
    smoke_config = PROJECT / 'configs/deeptune_smoke.yaml'
    smoke_config.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False), encoding='utf-8')
    run_streaming([
        sys.executable, '-u', str(PROJECT / 'train_dinov3_deeptune.py'),
        'all', '--backbone', BACKBONE, '--dataset-kind', DATASET_KIND,
        '--config', str(smoke_config), '--quick-smoke',
    ], PROJECT / 'deeptune_smoke.log')
    print('SMOKE COMPLETE | all three trainability stages passed.', flush=True)
else:
    print('Smoke test skipped explicitly.', flush=True)


## Start or continue deep fine-tuning

With `RESUME_CHECKPOINT = None`, initialization comes strictly from the selected attached safetensors. Epoch output is streamed once and also saved to `deeptune_training.log`.


In [ ]:
command = [
    sys.executable, '-u', str(PROJECT / 'train_dinov3_deeptune.py'),
    'all', '--backbone', BACKBONE, '--dataset-kind', DATASET_KIND,
    '--config', str(config_path),
]
if RESUME_CHECKPOINT is not None:
    resume_path = Path(RESUME_CHECKPOINT)
    if not resume_path.is_file():
        raise FileNotFoundError(f'Resume checkpoint not found: {resume_path}')
    command += ['--resume-checkpoint', str(resume_path)]

print(
    f'TRAINING START | {model_spec["display_name"]} on {data_spec["display_name"]} '
    '| 5 head + 5 partial + up to 100 full-backbone epochs',
    flush=True,
)
print('CHECKPOINT MONITOR | val_unseen Recall@1', flush=True)
run_streaming(command, PROJECT / 'deeptune_training.log')

summary_path = Path(cfg['runs_dir']) / RUN_NAME / 'run_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
required_metrics = {
    'accuracy', 'recall_at_1', 'recall_at_2', 'recall_at_5', 'recall_at_10',
    'mrr', 'median_rank', 'mean_rank', 'num_queries',
}
for split, values in summary['final_metrics'].items():
    missing = required_metrics - set(values)
    if missing:
        raise RuntimeError(f'{split} metrics are incomplete: {sorted(missing)}')
    assert abs(values['accuracy'] - values['recall_at_1']) < 1e-12
print(
    f'RESULT VERIFIED | backbone={summary["backbone"]} | '
    f'dataset={summary["dataset_kind"]} | epochs={summary["epochs_completed"]}',
    flush=True,
)
for split, values in summary['final_metrics'].items():
    print(
        f'{split}: accuracy/R@1={values["accuracy"]:.4f} '
        f'R@2={values["recall_at_2"]:.4f} R@5={values["recall_at_5"]:.4f} '
        f'R@10={values["recall_at_10"]:.4f}',
        flush=True,
    )
if summary['continuation_required']:
    print('Session budget reached: save outputs and continue from last.pt.', flush=True)
else:
    print('Deep fine-tuning completed. Best:', summary['best_checkpoint'], flush=True)
